In [ ]:
%matplotlib widget

import numpy as np
import matplotlib.pyplot as plt
from ipywidgets import FloatSlider, VBox, HBox, Layout, HTML, HTMLMath
from IPython.display import display

plt.ioff()

# ==============================================================================
# JUPYTER DISPLAY SETTINGS
# ==============================================================================

display(HTML("""
<style>
.jp-OutputArea,
.jp-OutputArea-child,
.jp-OutputArea-output {
    overflow: visible !important;
    max-height: none !important;
    height: auto !important;
}

.output,
.output_area,
.output_subarea,
.output_scroll {
    overflow: visible !important;
    max-height: none !important;
    height: auto !important;
}

.jupyter-widgets,
.widget-box,
.widget-html,
.widget-html-content {
    overflow: visible !important;
    max-height: none !important;
}

.jp-Cell-outputWrapper {
    overflow: visible !important;
}
</style>
"""))

# ==============================================================================
# USAGE
#
# This notebook demonstrates the Bode magnitude-phase relation for a causal,
# stable, minimum-phase continuous-time system.
#
# The example system is
#
#       H(s) = ωc / (s + ωc).
#
# The notebook performs two completely independent calculations of the phase:
#
# 1. The exact phase is calculated directly from H(jω).
#
# 2. The reconstructed phase is calculated ONLY from the logarithmic magnitude
#    ρ(ω) = ln|H(jω)| by means of the Bode integral
#
#       θ(ω0) = (1/π) ∫ [dρ(u)/du] W(u) du,
#
#    where
#
#       u = ln(ω/ω0)
#
#    and
#
#       W(u) = ln(coth(|u|/2)).
#
# The exact phase is NOT used in the reconstruction.
#
# The student should observe:
#
# - The first plot shows the logarithmic magnitude ρ(ω) used as the input
#   information for the reconstruction.
#
# - The second plot shows the Bode weighting function W(u). The weighting
#   function itself does not depend on Umax. The dashed vertical lines indicate
#   the finite integration limits -Umax and +Umax.
#
# - The third plot compares the exact phase (red solid curve) with the phase
#   reconstructed from the magnitude alone (black dashed curve).
#
# - Their close agreement demonstrates that, for a minimum-phase system,
#   the phase can be determined from the magnitude response.
#
# - Umax controls the finite interval [-Umax,+Umax] used to approximate the
#   theoretically infinite Bode integral. Increasing Umax generally reduces
#   the truncation error.
# ==============================================================================

usage = HTML("""
<div style="
    border:1px solid #9ec9f5;
    border-radius:7px;
    padding:8px 10px;
    margin:0px 0px 7px 0px;
    font-size:13px;
    line-height:1.45;
    background-color:#f7fbff;
    width:750px;
    max-width:750px;
    box-sizing:border-box;
">
<b>Purpose:</b>
Demonstrate the Bode magnitude-phase relation for a causal, stable, minimum-phase continuous-time system.
<br>
<b>Interpretation:</b>
The red phase curve is calculated directly from H(jω), whereas the black dashed curve is reconstructed <b>using only the magnitude response</b> through the Bode integral. Their close agreement shows that, for a minimum-phase system, the phase can be determined from the magnitude. Small differences are caused by the finite numerical integration interval.
</div>
""", layout=Layout(width='760px', max_width='760px'))

# ==============================================================================
# EQUATIONS
# ==============================================================================

formula_1 = HTMLMath(value=r"""
\[
H(s)=\frac{\omega_c}{s+\omega_c},
\qquad
\rho(\omega)=\ln|H(j\omega)|,
\qquad
\vartheta(\omega)=\angle H(j\omega)
\]
""", layout=Layout(width='750px', max_width='750px'))

formula_2 = HTMLMath(value=r"""
\[
\vartheta(\omega_0)=\frac{1}{\pi}\int_{-\infty}^{+\infty}
\frac{d\rho(u)}{du}
\ln\left(\coth\frac{|u|}{2}\right)\,du,
\qquad
u=\ln\frac{\omega}{\omega_0}
\]
""", layout=Layout(width='750px', max_width='750px'))

# ==============================================================================
# CONTROLS
# ==============================================================================

slider_layout = Layout(width='250px')
style_opts = {'description_width':'70px'}

wc_slider = FloatSlider(value=1.0, min=0.5, max=5.0, step=0.1, description='ωc:', continuous_update=True, readout_format='.1f', style=style_opts, layout=slider_layout)

umax_slider = FloatSlider(value=8.0, min=4.0, max=12.0, step=0.5, description='Umax:', continuous_update=True, readout_format='.1f', style=style_opts, layout=slider_layout)

parameter_title = HTML("""
<div style="font-size:14px; font-weight:bold; margin-top:3px; margin-bottom:5px;">
Parameters:
</div>
""")

info_html = HTML(layout=Layout(width='265px', max_width='265px'))

# ==============================================================================
# FIGURE 1: LOG-MAGNITUDE RESPONSE
# ==============================================================================

fig_mag, ax_mag = plt.subplots(figsize=(6.5, 2.4))

mag_line, = ax_mag.plot([], [], 'r-', linewidth=2.0, label='ρ(ω) = ln|H(jω)|')

ax_mag.set_xscale('log')
ax_mag.set_xlabel('Angular Frequency ω (rad/s)', fontsize=10)
ax_mag.set_ylabel('ρ(ω)', fontsize=10)
ax_mag.set_title('Log-Magnitude Response', fontsize=12, fontweight='bold', pad=5)
ax_mag.tick_params(axis='both', labelsize=9)
ax_mag.grid(True, which='both', linestyle=':', alpha=0.5)
ax_mag.legend(loc='center left', bbox_to_anchor=(1.01, 0.5), fontsize=9)

fig_mag.subplots_adjust(left=0.12, right=0.75, bottom=0.23, top=0.83)

fig_mag.canvas.header_visible = False
fig_mag.canvas.toolbar_visible = False
fig_mag.canvas.resizable = False
fig_mag.canvas.layout.width = '650px'
fig_mag.canvas.layout.height = '245px'
fig_mag.canvas.layout.margin = '0px 0px -6px 0px'

# ==============================================================================
# FIGURE 2: BODE WEIGHTING FUNCTION
# ==============================================================================

fig_weight, ax_weight = plt.subplots(figsize=(6.5, 2.4))

weight_line, = ax_weight.plot([], [], 'r-', linewidth=2.0, label='W(u)')

left_limit_line = ax_weight.axvline(-8.0, color='black', linestyle='--', linewidth=1.0, label='±Umax')
right_limit_line = ax_weight.axvline(8.0, color='black', linestyle='--', linewidth=1.0)

ax_weight.axvline(0.0, color='gray', linestyle=':', linewidth=1.0)

ax_weight.set_xlabel('Log-Frequency Distance  u = ln(ω/ω₀)', fontsize=10)
ax_weight.set_ylabel('W(u)', fontsize=10)
ax_weight.set_title('Bode Weighting Function', fontsize=12, fontweight='bold', pad=5)
ax_weight.tick_params(axis='both', labelsize=9)
ax_weight.grid(True, linestyle=':', alpha=0.5)
ax_weight.legend(loc='center left', bbox_to_anchor=(1.01, 0.5), fontsize=9)

fig_weight.subplots_adjust(left=0.12, right=0.75, bottom=0.23, top=0.83)

fig_weight.canvas.header_visible = False
fig_weight.canvas.toolbar_visible = False
fig_weight.canvas.resizable = False
fig_weight.canvas.layout.width = '650px'
fig_weight.canvas.layout.height = '245px'
fig_weight.canvas.layout.margin = '0px 0px -6px 0px'

# ==============================================================================
# FIGURE 3: EXACT AND RECONSTRUCTED PHASE
# ==============================================================================

fig_phase, ax_phase = plt.subplots(figsize=(6.5, 2.4))

phase_exact_line, = ax_phase.plot([], [], 'r-', linewidth=2.0, label='Exact phase')
phase_bode_line, = ax_phase.plot([], [], 'k--', linewidth=1.7, label='Reconstructed from magnitude')

ax_phase.set_xscale('log')
ax_phase.set_xlabel('Angular Frequency ω (rad/s)', fontsize=10)
ax_phase.set_ylabel('Phase (rad)', fontsize=10)
ax_phase.set_title('Phase Reconstruction from Magnitude', fontsize=12, fontweight='bold', pad=5)
ax_phase.tick_params(axis='both', labelsize=9)
ax_phase.grid(True, which='both', linestyle=':', alpha=0.5)
ax_phase.axhline(0.0, color='gray', linewidth=0.8)
ax_phase.legend(loc='center left', bbox_to_anchor=(1.01, 0.5), fontsize=9)

fig_phase.subplots_adjust(left=0.12, right=0.75, bottom=0.23, top=0.83)

fig_phase.canvas.header_visible = False
fig_phase.canvas.toolbar_visible = False
fig_phase.canvas.resizable = False
fig_phase.canvas.layout.width = '650px'
fig_phase.canvas.layout.height = '245px'

# ==============================================================================
# NUMERICAL BODE PHASE RECONSTRUCTION
# ==============================================================================

def reconstruct_phase_from_magnitude(omega_eval, wc, umax, n_u=1600):

    du = 2.0 * umax / n_u
    u = -umax + (np.arange(n_u) + 0.5) * du

    omega_matrix = omega_eval[:, None] * np.exp(u)[None, :]

    rho_matrix = -0.5 * np.log(1.0 + (omega_matrix / wc)**2)

    drho_du = np.gradient(rho_matrix, du, axis=1, edge_order=2)

    weight = np.log(1.0 / np.tanh(np.abs(u) / 2.0))

    phase_reconstructed = (du / np.pi) * np.sum(drho_du * weight[None, :], axis=1)

    return phase_reconstructed

# ==============================================================================
# UPDATE FUNCTION
# ==============================================================================

def update_plots(change=None):

    wc = wc_slider.value
    umax = umax_slider.value

    omega = np.geomspace(0.05, 20.0, 260)

    # --------------------------------------------------------------------------
    # Exact response
    # --------------------------------------------------------------------------

    H = wc / (wc + 1j * omega)

    rho_exact = np.log(np.abs(H))
    phase_exact = np.angle(H)

    # --------------------------------------------------------------------------
    # Phase reconstructed ONLY from magnitude
    # --------------------------------------------------------------------------

    phase_reconstructed = reconstruct_phase_from_magnitude(omega, wc, umax)

    # --------------------------------------------------------------------------
    # Bode weighting function
    # --------------------------------------------------------------------------

    u_plot_max = max(6.0, umax + 0.5)

    u_display = np.linspace(-u_plot_max, u_plot_max, 1600)

    mask = np.abs(u_display) > 0.015

    weight_display = np.full_like(u_display, np.nan)

    weight_display[mask] = np.log(1.0 / np.tanh(np.abs(u_display[mask]) / 2.0))

    # --------------------------------------------------------------------------
    # Update log-magnitude plot
    # --------------------------------------------------------------------------

    mag_line.set_data(omega, rho_exact)

    ax_mag.set_xlim(0.05, 20.0)

    rho_min = np.min(rho_exact)
    rho_max = np.max(rho_exact)
    rho_margin = 0.08 * max(rho_max - rho_min, 1.0)

    ax_mag.set_ylim(rho_min - rho_margin, rho_max + rho_margin)

    # --------------------------------------------------------------------------
    # Update weighting plot
    # --------------------------------------------------------------------------

    weight_line.set_data(u_display, weight_display)

    left_limit_line.set_xdata([-umax, -umax])
    right_limit_line.set_xdata([umax, umax])

    ax_weight.set_xlim(-u_plot_max, u_plot_max)
    ax_weight.set_ylim(0.0, 5.0)

    # --------------------------------------------------------------------------
    # Update phase plot
    # --------------------------------------------------------------------------

    phase_exact_line.set_data(omega, phase_exact)
    phase_bode_line.set_data(omega, phase_reconstructed)

    ax_phase.set_xlim(0.05, 20.0)

    phase_min = min(np.min(phase_exact), np.min(phase_reconstructed))
    phase_max = max(np.max(phase_exact), np.max(phase_reconstructed))

    phase_margin = 0.08 * max(phase_max - phase_min, 0.5)

    ax_phase.set_ylim(phase_min - phase_margin, phase_max + phase_margin)

    # --------------------------------------------------------------------------
    # Numerical error
    # --------------------------------------------------------------------------

    phase_error = np.max(np.abs(phase_exact - phase_reconstructed))

    # --------------------------------------------------------------------------
    # Information panel
    # --------------------------------------------------------------------------

    info_html.value = f"""
    <div style="
        border:1px solid #cccccc;
        border-radius:7px;
        padding:8px 9px;
        margin-top:9px;
        font-size:12px;
        line-height:1.65;
        background:white;
        width:260px;
        box-sizing:border-box;
    ">

    <div>
        <b>System:</b>
        <span style="color:#0066cc;">H(s) = ωc/(s+ωc)</span>
    </div>

    <div>
        <b>ωc:</b>
        <span style="color:#0066cc;">{wc:.2f} rad/s</span>
    </div>

    <div>
        <b>Integration range:</b>
        <span style="color:#0066cc;">[−{umax:.1f}, +{umax:.1f}] in u</span>
    </div>

    <div style="
        margin-top:6px;
        padding-top:6px;
        border-top:1px solid #eeeeee;
    ">
        <b>Exact quantities:</b><br>
        <span style="color:#0066cc;">
        ρ(ω) = ln|H(jω)|<br>
        θ(ω) = −tan⁻¹(ω/ωc)
        </span>
    </div>

    <div style="
        margin-top:6px;
        padding-top:6px;
        border-top:1px solid #eeeeee;
    ">
        <b>Maximum phase error:</b><br>
        <span style="color:#0066cc;">{phase_error:.4e} rad</span>
    </div>

    <div style="
        margin-top:6px;
        padding-top:6px;
        border-top:1px solid #eeeeee;
    ">
        <b>Key observation:</b><br>
        The dashed phase curve is reconstructed from the
        <b>magnitude response only</b>. The exact phase is used
        solely for comparison.
    </div>

    </div>
    """

    fig_mag.canvas.draw_idle()
    fig_weight.canvas.draw_idle()
    fig_phase.canvas.draw_idle()

# ==============================================================================
# CALLBACKS
# ==============================================================================

wc_slider.observe(update_plots, names='value')
umax_slider.observe(update_plots, names='value')

# ==============================================================================
# LAYOUT
# ==============================================================================

controls = VBox([parameter_title, wc_slider, umax_slider, info_html], layout=Layout(width='275px', min_width='275px', max_width='275px', flex='0 0 275px', align_items='flex-start'))

plots = VBox([fig_mag.canvas, fig_weight.canvas, fig_phase.canvas], layout=Layout(width='660px', min_width='660px', max_width='660px', align_items='flex-start'))

main_layout = HBox([controls, plots], layout=Layout(width='935px', min_width='935px', max_width='935px', align_items='flex-start', justify_content='flex-start'))

# ==============================================================================
# INITIAL DISPLAY
# ==============================================================================

update_plots()

display(usage)
display(formula_1)
display(formula_2)
display(main_layout)